In [16]:
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import time

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("GPT_API_KEY")
client = OpenAI(api_key=key)

In [5]:
LABELS = [
    "Neutral/Factual",
    "Biased/Leading",
    "Promotional",
    "Sensational/Clickbait",
    "Human-interest"
]

RULES = """
Classify the headline into EXACTLY ONE label.

Neutral/Factual:
- Straight reporting
- Deadlines, policies, events, announcements
- No emotional or persuasive language

Biased/Leading:
- Emotional framing
- Opinionated or suggestive tone
- Quotes used to provoke reaction

Promotional:
- Advertising, marketing, courses, brands, launches
- Buzzwords like "lead", "next wave", "taps into"

Sensational/Clickbait:
- Curiosity hooks, teasers
- Dramatic or exaggerated phrasing
- Question headlines designed to provoke clicks

Human-interest:
- Personal struggles or achievements
- Emotional life stories
"""

In [18]:
# Headlines Labeling Function

def label_headline(headline, country):
    prompt = f"""
You are a professional media bias analyst.

{RULES}

Headline: "{headline}"

Country: {country}

Return ONLY in this format (no extra text):

Label: <one label from the list>
Reason: <one short sentence>
Confidence: <number between 0 and 1>
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        text = response.choices[0].message.content.strip()

        label = text.split("Label:")[1].split("\n")[0].strip()
        reason = text.split("Reason")[1].split("\n")[0].strip()
        confidence = float(text.split("Confidence:")[1].strip())

        if label not in LABELS:
            return "ERROR",  "Invalid label returned", 0.0

        return label, response, confidence

    except Exception as e:
        return "ERROR", str(e), 0.0

In [22]:
# Load CSV data

# df = pd.read_csv(r"../raw/1000-US-Ind-Aus.csv")
df = pd.read_csv(r"../raw/headlines_raw_20260129_1555.csv")
df.head()

,headline,country,label
0,SpaceX launches advanced GPS satellite for US ...,USA,NaN
1,South Korea’s former first lady sentenced to 2...,USA,NaN
2,‘A betrayal’: Southwest’s new plus size policy...,USA,NaN
3,2026 NFL Draft: 8 prospects who stood out in E...,USA,NaN
4,How should Chelsea line up against Napoli with...,USA,NaN


In [23]:
# Run Labeling

labels, reasons, confidences = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    label, reason, confidence = label_headline(
        row["headline"],
        row["country"]
    )

    labels.append(label)
    reasons.append(reason)
    confidences.append(confidence)

    time.sleep(0.4) # rate-limit safety

100%|██████████| 27/27 [01:12<00:00,  2.70s/it]


In [24]:
# Save Results

df["label"] = labels
df["reason"] = reasons
df["confidence"] = confidence

df.to_csv("headlines_labeled.csv", index=False)

In [25]:
# Quality Control

df[df["confidence"] < 0.6].sample(10)

,headline,country,label,reason,confidence
14,Asia markets open mostly higher after S&P 500 ...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
12,Diplomats Worry Xi’s Purge Will Curb Critical ...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
1,South Korea’s former first lady sentenced to 2...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
21,Breaking Away From the Pack: The Case for the ...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
25,Marburg virus disease - Ethiopia - World Healt...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
17,Citi seeks to move harassment claim against to...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
15,Samsung’s first Galaxy S26 teaser is for the ‘...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
24,William Foege - Legacy | Obituary,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
9,Consumer Price Index climbs 3.6% YoY in Decemb...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0
8,After Further Review: How the Patriots Defense...,USA,ERROR,Error code: 429 - {'error': {'message': 'You e...,0.0


In [26]:
# Label Distribution

df["label"].value_counts()

label
ERROR    27
Name: count, dtype: int64

You are a professional media bias analyst.

{Classify the headline into EXACTLY ONE label.

Neutral/Factual:
- Straight reporting
- Deadlines, policies, events, announcements
- No emotional or persuasive language

Biased/Leading:
- Emotional framing
- Opinionated or suggestive tone
- Quotes used to provoke reaction

Promotional:
- Advertising, marketing, courses, brands, launches
- Buzzwords like "lead", "next wave", "taps into"

Sensational/Clickbait:
- Curiosity hooks, teasers
- Dramatic or exaggerated phrasing
- Question headlines designed to provoke clicks

Human-interest:
- Personal struggles or achievements
- Emotional life stories}

LABELS: "Neutral/Factual",
    "Biased/Leading",
    "Promotional",
    "Sensational/Clickbait",
    "Human-interest"

DATA: "

"

Return ONLY in this format (no extra text):

Label: <one label from the list>
Reason: <one short sentence>
Confidence: <number between 0 and 1>